# Phase 12 — Jacobi-Linse an Position 43

Liest, **wozu ein Zustand das Modell zu sagen disponiert**: Residuum aus
Schicht *l* linear in die Endschicht-Basis transportiert, mit dem Unembedding
dekodiert. Statt die 2048×2048-Jacobimatrix aufzustellen (2048 Rückwärtspässe
pro Prompt) rechnen wir nur ihre Wirkung auf die interessierenden Vektoren —
zwei Rückwärtspässe pro Prompt. Die Zelle beweist die Gleichwertigkeit gegen
`jacobian_for_prompt` aus dem Repo, bevor sie GPU-Zeit verbraucht.

19 Positionen × 9 Voll-Attention-Layer, Zielgröße Fremdschrift-Masse,
Kontrolle ist die Logit-Linse an derselben Stelle.

Selbstversorgend — **frische Runtime**, dann nur diese Zelle. ~10 min.

In [ ]:
# === JACOBI-LINSE an Position 43: wozu ist der Zustand disponiert? ========
# Die Jacobi-Linse (Anthropic, "Verbalizable Representations Form a Global
# Workspace in Language Models") liest, WAS EIN ZUSTAND DAS MODELL ZU SAGEN
# DISPONIERT. Sie transportiert ein Residuum h aus Schicht l linear in die
# Endschicht-Basis und dekodiert es mit dem Unembedding des Modells:
#     lens_l(h) = unembed( J_l @ h ),   J_l = E[ dh_final / dh_l ]
# gemittelt ueber einen Korpus. Das ist genau die Frage, an der der RDX-Zeuge
# haengt: ist die Disposition am Koeder-Token ueberhaupt lesbar - und zwar als
# Vokabular, nicht als Cluster.
#
# Machbarkeit. Der Referenzcode stellt J_l explizit auf: d_model Rueckwaerts-
# paesse pro Prompt, bei 2048 Dimensionen und 100 Prompts sind das Groessen-
# ordnungen jenseits einer Colab-Karte. Wir brauchen die Matrix aber nie -
# nur ihre WIRKUNG auf eine Handvoll Vektoren. Das liefert ein Vorwaerts-
# Differential:
#     g(s) = d(y.s)/du = J^T s   (ein Rueckwaertspass, create_graph)
#     d(g.v)/ds        = J v     (ein zweiter)
# Kosten: ein Vorwaertspass und zwei Rueckwaertspaesse pro Prompt statt 2048.
# Die Reduktion ist dieselbe wie im Original (Tangente an allen gueltigen
# Quellpositionen, Ausgabe ueber alle gueltigen Zielpositionen summiert,
# geteilt durch deren Anzahl) - und die Zelle BEWEIST das, bevor sie GPU-Zeit
# verbrennt: sie laesst jacobian_for_prompt aus dem Repo auf dem Mini-Modell
# aus tests/tiny.py laufen und vergleicht J@v mit unserem JVP. Weicht es ab,
# bricht ein assert ab.
#
# Zweite Huerde: durch ein FP8-quantisiertes MoE muss ueberhaupt erst ein
# Rueckwaertspass laufen. Die Zelle testet das explizit und faellt sonst
# sauber auf die Logit-Linse zurueck, statt mit einem Kernel-Fehler zu sterben.
#
# Gelesen wird an 19 Positionen des Tabellen-Prompts, darunter K und Q der
# Koeder-Phrase, ueber neun Voll-Attention-Layer. Zielgroesse ist die
# Fremdschrift-Masse in der Linsen-Verteilung; Kontrolle ist die LOGIT-Linse
# an derselben Stelle. Der interessante Fall ist die Luecke: eine Disposition,
# die der Transport zeigt und das rohe Residuum nicht.
#
# Vorregistrierung Nr. 28: KEINE ~40%, RICHTUNG ~35%, BEIDE ~15%,
# NICHT ANWENDBAR (kein Gradient) ~10%.
# Selbstversorgend, FRISCHE Runtime, einzige Zelle. ~10 min.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN",'jacobian_lens')
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)

import matplotlib.pyplot as plt
import subprocess, sys
N_FIT=32; MAXTOK=96; BATCH=5; TOPK=6; SEED=0
LAYERS=[3,7,11,15,19,23,27,31,35]     # Voll-Attention-Layer als Quellschichten
N_CTRL=15                              # Kontrollpositionen neben den Koeder-Tokens
MASK_NPZ=(glob.glob("/content/drive/MyDrive/**/vocab_foreign_masks.npz",recursive=True) or [""])[0]
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- reine Hilfslogik (offline testbar) ------------------------
def sample_positions(lo,hi,n,must,seed=0):
    """gleichmaessiges Raster ueber [lo,hi] plus Pflichtpositionen, sortiert"""
    if hi<=lo: return sorted(set(must))
    step=max(1,(hi-lo)//max(1,n))
    grid=list(range(lo,hi+1,step))[:n]
    return sorted(set(grid+[t for t in must if lo<=t<=hi]))
def chunks(xs,k):
    return [xs[i:i+k] for i in range(0,len(xs),k)]
def rank_of(vals,idx):
    """Rang von vals[idx] unter allen Werten, 1 = groesster"""
    v=list(vals); target=v[idx]
    return 1+sum(1 for x in v if x>target)
def verdict_jlens(rank_q,n_pos,best_layer,mass_q,mass_ctrl,logit_rank_q):
    """RICHTUNG: die Jacobi-Linse zeigt am Koeder-Token eine Fremdschrift-
       Disposition, die die Logit-Linse dort NICHT zeigt.
       BEIDE: beide Linsen zeigen sie - der Transport fuegt nichts hinzu.
       KEINE: das Koeder-Token ragt nicht heraus."""
    top=rank_q<=max(2,n_pos//10)
    if not top: return "KEINE"
    if logit_rank_q<=max(2,n_pos//10): return "BEIDE"
    return "RICHTUNG"
# ---------------- jlens beschaffen -----------------------------------------
JL="/content/jacobian_lens"
if not os.path.isdir(os.path.join(JL,"jlens")):
    r=subprocess.run(["git","clone","--depth","1",
                      "https://github.com/Erikiss/jacobian-lens",JL],
                     capture_output=True,text=True,timeout=600)
    assert r.returncode==0, "Klon fehlgeschlagen: %s"%r.stderr.strip()[-200:]
subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps","-e",JL],
               capture_output=True,text=True)
if JL not in sys.path: sys.path.insert(0,JL)
from jlens.hooks import ActivationRecorder
from jlens.fitting import jacobian_for_prompt, valid_position_mask
print("jlens geladen aus", JL)
# ---------------- der Kern: Transport OHNE die Matrix zu bauen --------------
def jvp_transport(fwd,blocks,input_ids,source_layers,target_layer,V,skip_first,B):
    """Liefert {l: J_l @ v} fuer je B Vektoren gleichzeitig - ohne J_l je
       aufzustellen. Der Upstream baut die 2048x2048-Matrix mit d_model
       Rueckwaertspaessen pro Prompt; wir brauchen nur ihre WIRKUNG auf ein
       paar Vektoren, und die liefert ein Vorwaerts-Differential (JVP), das
       hier ueber den Doppel-Rueckwaerts-Trick gerechnet wird:
           g(s) = d(y.s)/du = J^T s      (ein Rueckwaertspass, create_graph)
           d(g.v)/ds        = J v        (ein zweiter Rueckwaertspass)
       Reduktion identisch zum Upstream-Schaetzer: Tangente an ALLEN gueltigen
       Quellpositionen, Ausgabe ueber alle gueltigen Zielpositionen summiert,
       durch deren Anzahl geteilt."""
    with ActivationRecorder(blocks,at=[*source_layers,target_layer],
                            start_graph_at=min(source_layers)) as rec, torch.enable_grad():
        ids=input_ids.expand(B,-1)
        fwd(ids)
        y=rec.activations[target_layer]
        us=[rec.activations[l] for l in source_layers]
        pm=valid_position_mask(ids.shape[1],skip_first=skip_first)
        pos=pm.nonzero(as_tuple=True)[0].to(y.device)
        npos=int(pm.sum())
        s=torch.zeros_like(y,requires_grad=True)
        gs=torch.autograd.grad(y,us,grad_outputs=s,create_graph=True)
        out={}
        for i,l in enumerate(source_layers):
            t=torch.zeros_like(us[i])
            p2=pos.to(t.device)
            t[:,p2,:]=V[l].to(t.dtype).to(t.device)[:,None,:]
            jv,=torch.autograd.grad(gs[i],s,grad_outputs=t,
                                    retain_graph=(i<len(source_layers)-1))
            out[l]=jv[:,pos,:].float().sum(1)/npos
        return out
# ---------------- Gleichwertigkeitsbeweis gegen den Upstream ---------------
from tests.tiny import TinyDecoder
_t=TinyDecoder(n_layers=5,d_model=8,seed=0).eval()
for _p in _t.parameters(): _p.requires_grad_(False)
_TXT="the quick brown fox jumps over the lazy dog again and again"
_SL=[0,1,2]; _SK=2
_J,_sq,_nv=jacobian_for_prompt(_t,_TXT,_SL,dim_batch=4,max_seq_len=64,skip_first=_SK)
_ids=_t.encode(_TXT,max_length=64)
torch.manual_seed(0); _V={l:torch.randn(3,8) for l in _SL}
_JV=jvp_transport(lambda x:_t(x),_t.layers,_ids,_SL,_t.n_layers-1,_V,_SK,3)
_dev=max(float((_JV[l]-(_V[l]@_J[l].T)).abs().max()) for l in _SL)
_sc=max(float(_JV[l].abs().max()) for l in _SL)
print("Gleichwertigkeit zum Upstream-Schaetzer: max|JVP - J@v| = %.3e (Skala %.3e)"
      %(_dev,_sc))
assert _dev<1e-4*max(_sc,1.0), ("JVP stimmt NICHT mit jacobian_for_prompt ueberein - "
                                "Abbruch, bevor GPU-Zeit verbrannt wird")
print("  -> bestaetigt: der Transport ist derselbe, nur ohne die Matrix.")
del _t,_J,_JV
gc.collect()
# ---------------- Differenzierbarkeit des echten Modells --------------------
BLOCKS=model.model.layers
for _p in model.parameters(): _p.requires_grad_(False)
def fwd(ids): return model.model(input_ids=ids)
_UD=next(model.model.norm.parameters()).dtype
def unembed(r):
    return model.lm_head(model.model.norm(r.to(_UD)))
try:
    _ii=tokenizer("probe text for gradients",return_tensors="pt").input_ids.to(model.device)
    with ActivationRecorder(BLOCKS,at=[3,model.config.num_hidden_layers-1],
                            start_graph_at=3) as _r, torch.enable_grad():
        fwd(_ii)
        _g,=torch.autograd.grad(_r.activations[model.config.num_hidden_layers-1].sum(),
                                _r.activations[3])
    DIFFBAR=bool(torch.isfinite(_g).all() and _g.abs().sum()>0)
    print("Differenzierbarkeit des Modells: %s (|grad|=%.3e)"
          %("ja" if DIFFBAR else "NEIN - Gradienten null/nicht endlich",float(_g.abs().mean())))
except Exception as e:
    DIFFBAR=False
    print("Differenzierbarkeit des Modells: NEIN - %s: %s"%(type(e).__name__,str(e)[:120]))
del _g
gc.collect(); torch.cuda.empty_cache()
# ---------------- Zielprompt und Positionen --------------------------------
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
pre=think_prefix(TAB)
enc=tokenizer(pre,return_offsets_mapping=True); IDS=enc["input_ids"]; L=len(IDS)
c0=len(SCAFF)+TAB.index("local name"); c1=c0+len("local name")
DEC=[i for i,(a,b) in enumerate(enc["offset_mapping"]) if b>c0 and a<c1 and b>a]
K,Q=DEC[0],DEC[-1]
UT=[i for i,(a,b) in enumerate(enc["offset_mapping"])
    if b>len(SCAFF) and a<len(SCAFF)+len(TAB) and b>a]
POS=sample_positions(UT[0],UT[-1],N_CTRL,[K-1,K,Q,Q+1],seed=SEED)
jQ=POS.index(Q)
print("\nZielprompt %d Tokens | Koeder K=%d %r  Q=%d %r | %d Lesepositionen"
      %(L,K,tokenizer.decode([IDS[K]]),Q,tokenizer.decode([IDS[Q]]),len(POS)))
ids_t=torch.tensor([IDS],device=model.device)
with torch.no_grad():
    HS=model(input_ids=ids_t,output_hidden_states=True).hidden_states
H={l:torch.stack([HS[l+1][0,p] for p in POS]).float().cpu() for l in LAYERS}
del HS
gc.collect(); torch.cuda.empty_cache()
# ---------------- Korpus fuer die Mittelung ---------------------------------
rng=np.random.default_rng(SEED)
corp=[p for p in PROMPT_IDS if 200<len(PROMPTS[p])<=1200]
corp=[PROMPTS[corp[i]] for i in rng.permutation(len(corp))[:N_FIT]]
print("Korpus fuer die Jacobi-Mittelung: %d Prompts, je bis %d Tokens"%(len(corp),MAXTOK))
# ---------------- Transport -------------------------------------------------
TGT=model.config.num_hidden_layers-1
ACC={l:torch.zeros(len(POS),model.config.hidden_size) for l in LAYERS}
NOK=0
if DIFFBAR:
    grp=chunks(list(range(len(POS))),BATCH)
    for ci,ctext in enumerate(corp):
        cid=tokenizer(ctext,return_tensors="pt",truncation=True,
                      max_length=MAXTOK).input_ids.to(model.device)
        if cid.shape[1]<24: continue
        try:
            for g in grp:
                V={l:H[l][g].to(model.device) for l in LAYERS}
                out=jvp_transport(fwd,BLOCKS,cid,LAYERS,TGT,V,16,len(g))
                for l in LAYERS: ACC[l][g]+=out[l].cpu()
                del out,V
            NOK+=1
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print("  OOM bei Prompt %d - uebersprungen (BATCH kleiner setzen)"%ci)
        gc.collect(); torch.cuda.empty_cache()
        if (ci+1)%8==0: print("  %2d/%d Korpus-Prompts"%(ci+1,len(corp)))
    for l in LAYERS: ACC[l]/=max(NOK,1)
    print("  gemittelt ueber %d Prompts"%NOK)
# ---------------- Auslesen: Jacobi-Linse gegen Logit-Linse ------------------
assert MASK_NPZ and os.path.exists(MASK_NPZ), "vocab_foreign_masks.npz nicht gefunden"
M_script=torch.tensor(np.load(MASK_NPZ)["script"])
def fmass(logits):
    p=torch.softmax(logits.float(),-1); V=p.shape[-1]
    m=M_script.to(p.device)
    if m.shape[0]<V: m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
    return p[...,m[:V]].sum(-1)
JAC=np.zeros((len(LAYERS),len(POS))); LOG=np.zeros_like(JAC); TOPS={}
with torch.no_grad():
    for i,l in enumerate(LAYERS):
        lg_j=unembed(ACC[l].to(model.device)) if DIFFBAR else None
        lg_l=unembed(H[l].to(model.device))
        LOG[i]=fmass(lg_l).cpu().numpy()
        if lg_j is not None:
            JAC[i]=fmass(lg_j).cpu().numpy()
            TOPS[l]=[tokenizer.decode([t]) for t in lg_j[jQ].topk(TOPK).indices.tolist()]
        TOPS.setdefault(("logit",l),[tokenizer.decode([t]) for t in lg_l[jQ].topk(TOPK).indices.tolist()])
        del lg_j,lg_l
if DIFFBAR:
    print("\nWas die Jacobi-Linse am Koeder-Token Q=%d liest (Top-%d):"%(Q,TOPK))
    for l in LAYERS:
        print("  L%-2d  %s"%(l," | ".join(repr(t) for t in TOPS[l])))
print("\nZum Vergleich die Logit-Linse an derselben Stelle:")
for l in LAYERS:
    print("  L%-2d  %s"%(l," | ".join(repr(t) for t in TOPS[("logit",l)])))
# ---------------- Statistik: ragt Q heraus? ---------------------------------
BEST=int(np.argmax(JAC[:,jQ])) if DIFFBAR else int(np.argmax(LOG[:,jQ]))
rq=rank_of(JAC[BEST],jQ) if DIFFBAR else len(POS)
rl=rank_of(LOG[BEST],jQ)
print("\nFremdschrift-Masse im Linsen-Auslesen (Maximum ueber Layer bei L%d):"%LAYERS[BEST])
if DIFFBAR:
    print("  Jacobi: Q=%.4f | Median der %d Positionen %.4f | Rang von Q: %d/%d"
          %(JAC[BEST,jQ],len(POS),float(np.median(JAC[BEST])),rq,len(POS)))
print("  Logit : Q=%.4f | Median %.4f | Rang von Q: %d/%d"
      %(LOG[BEST,jQ],float(np.median(LOG[BEST])),rl,len(POS)))
# ---------------- Karten ----------------------------------------------------
fig,axs=plt.subplots(1,2 if DIFFBAR else 1,figsize=(15 if DIFFBAR else 8,4.4),squeeze=False)
for ax,(M,name) in zip(axs[0],([(JAC,"Jacobi-Linse"),(LOG,"Logit-Linse")] if DIFFBAR
                               else [(LOG,"Logit-Linse")])):
    im=ax.imshow(np.log10(np.maximum(M,1e-12)),aspect="auto",cmap="magma",origin="lower")
    ax.set_yticks(range(len(LAYERS))); ax.set_yticklabels(["L%d"%l for l in LAYERS],fontsize=8)
    ax.set_xticks(range(len(POS)))
    ax.set_xticklabels(["%d %s"%(p,tokenizer.decode([IDS[p]]).strip()[:7]) for p in POS],
                       rotation=90,fontsize=7)
    ax.axvline(jQ,color="#22D3EE",lw=1.6)
    ax.set_title("%s: log10 Fremdschrift-Masse\n(tuerkis = Koeder-Token Q=%d)"%(name,Q),fontsize=10)
    plt.colorbar(im,ax=ax,fraction=.03,pad=.02)
plt.tight_layout(); plt.show()
# ---------------- Verdikt ---------------------------------------------------
code=verdict_jlens(rq,len(POS),LAYERS[BEST],JAC[BEST,jQ] if DIFFBAR else 0.0,
                   float(np.median(JAC[BEST])) if DIFFBAR else 0.0,rl)
print("\nVERDIKT:",end=" ")
if not DIFFBAR:
    print("NICHT ANWENDBAR: durch dieses Modell laeuft kein Rueckwaertspass")
    print("  (FP8-Quantisierung oder ein nicht differenzierbarer Kernel). Die")
    print("  Jacobi-Linse braucht Gradienten; nur die Logit-Linse ist gelaufen.")
elif code=="RICHTUNG":
    print("VERBALISIERBAR: die Jacobi-Linse liest am Koeder-Token eine Fremdschrift-")
    print("  Disposition (Rang %d von %d Positionen bei L%d), die Logit-Linse an"%(rq,len(POS),LAYERS[BEST]))
    print("  derselben Stelle nicht (Rang %d). Der Zustand ist also darauf gerichtet,"%rl)
    print("  etwas zu SAGEN, was im Residuum selbst noch nicht steht - genau der")
    print("  Zeuge, den das Modell nicht schreibt und darum nicht abkoppeln kann.")
elif code=="BEIDE":
    print("BEIDE LINSEN: die Disposition steht schon im Residuum (Logit-Rang %d),"%rl)
    print("  der Jacobi-Transport bestaetigt sie (Rang %d), fuegt aber nichts hinzu."%rq)
    print("  Der Zeuge steht - er braucht den Transport nicht.")
else:
    print("KEIN AUSSCHLAG AM KOEDER: Q=%d ragt in keiner Linse heraus (Jacobi-Rang"%Q)
    print("  %d, Logit-Rang %d von %d). Die Disposition ist an dieser Position"%(rq,rl,len(POS)))
    print("  nicht als Vokabular lesbar - passend zu Cell 19 und zum RDX-Befund.")
print("(Deskriptiv, ein Zielprompt: die Raenge sind ueber %d Positionen desselben"%len(POS))
print(" Prompts gerechnet, kein Test ueber Prompts hinweg.)")
JLENS_RESULTS=dict(verdict=code,diffbar=DIFFBAR,layers=LAYERS,positions=POS,Q=Q,K=K,
                   jac=JAC.tolist(),logit=LOG.tolist(),n_corpus=NOK,
                   rank_jac=rq,rank_logit=rl,best_layer=LAYERS[BEST])
wc_save_all()
